# eye_personal.ipynb — 내 눈 사진과 비교하는 개인 기준 모델 (11/05)

- **왜:** 지금 CNN 은 처음 보는 사람의 눈 한 장만 보고 판단 → 원래 눈이 가는 사람 · 반쯤 감은 눈에서 헷갈림. 숫자(CNN 확률 · EAR)로 만든 개인 기준은 효과 없음 (11/05 PC 시뮬레이션: 뜬 눈만 · 뜬 눈 + 감은 눈 2점 모두, 같은 오경보에서 0.5 ~ 2%p)
- **아이디어:** 운행 전에 **내 뜬 눈 5장 + 내 감은 눈 2장** 을 찍어 두고, 모델이 지금 눈을 **그 사진들과 비교**해서 판단 (처음 보는 사람이어도 그 사람의 기준 사진이 있음)
  - 학습: 548 학습용 사람들로 '문제' 를 계속 만들어 풂 — 한 사람의 눈 하나(문제) + 같은 사람 · 같은 쪽 눈의 뜬 눈 5장 · 감은 눈 2장(기준) → 감음 쪽인가?
  - 시험: 학습에 없던 사람 (548 학습용 val 115명) 에게 기준 사진만 주고 판단
- **반쯤 감은 눈은 학습 라벨 1 (감음 쪽)** — 목표는 "폰이 의심해 서버로 온 반쯤 감은 사진은 취소하지 않는 것" (11/05 팀장 문구). '경고해야 하는 눈' 으로 일반화하지 않음 (앱에서 반쯤은 1.5초 미세수면 대상이 아니고, 폰이 놓친 사진은 서버로 오지 않음). 감은 눈과 따로 보고

| 모델 | 보는 것 | 뜻 |
|---|---|---|
| `none` | 지금 눈만 | **같은 학습 방식의 기준** (반쯤 감은 눈 학습 효과만) |
| `open` | 지금 눈 + 내 뜬 눈 5장 | 지금 앱처럼 뜬 눈만 잴 때 |
| `open_closed` | 지금 눈 + 내 뜬 눈 5장 + 내 감은 눈 2장 | 감은 눈까지 잴 때 |

- 셋 다 **같은 데이터 · 같은 학습 방식 · 같은 시드** → 차이는 '기준 사진' 효과만. 지금 서버 모델(`eye_cnn12_k100_100_s0.pt` · `eye_cnn_548.pt`) 도 같은 사진으로 같이 잼 (참고)

### 0. 미리 정한 규칙 (결과 보기 전, 11/05)

1. 고르기 · 비교 = **548 학습용 val 115명** (학습에 없는 사람). 사람 · 눈 쪽마다 기준 사진 고정 (시드 0: 정면 뜬 눈 5장 + 감은 눈 2장) → 그 사진은 채점에서 뺌
2. 사진 단위 (두 눈 평균) — 서버처럼. 지표 = **같은 오경보(1 · 2 · 3 · 5%)에서 반쯤 감은 사진을 감음 쪽으로 못 본 비율 · 감은 눈을 감음 쪽으로 못 본 비율** (CNN 만, EAR 없음 — 서버에서는 '취소' 에 해당)
3. `open_closed` 가 **오경보 3% 에서 `none` 보다 반쯤 못 봄이 2%p 넘게 낮고, 감은 눈 못 봄이 `none` 이하**일 때만 "효과 있음" → 그때만 서버 · 앱에 넣는 방법을 팀에 제안 (앱 측정 화면에 '눈 감기 2초' 추가 + 서버가 기사별 기준 사진 보관 — C · 팀장 결정 필요)
4. 효과 있으면 최종 시험은 **새 데이터 한 번** (548 `TS_G1_2` 처음 쓰는 사람, 받는 중) — 결과가 나빠도 다시 고르지 않음
5. 151명 (Validation) · 팀 사진은 여기서 안 씀

- **실행 (Colab T4):** 위에서부터 차례로. 왼쪽 파일 탭에 `eye_preprocess.py` 를 `/content` 에 올리기. 드라이브 `sleep_detector/aihub548_eye/` (7-1 과 같은 파일) 필요
  - 시간: 모델 하나 약 15 ~ 25분 × 3 ≈ **1 ~ 1시간 15분**. 끊기면 1 ~ 3 다시 → 4 (끝난 에포크 · 모델은 건너뜀)
  - **eye_cnn.ipynb 11번과 같은 Colab 에서 동시에 X** (GPU 하나) — 11번 끝난 뒤

### 1. 준비 — 드라이브 · 설정 · 548 눈 사진 읽기 (반쯤 감은 눈 포함)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, datetime, importlib
import numpy as np, pandas as pd, cv2, torch, torchvision

assert os.path.exists('/content/eye_preprocess.py'), '왼쪽 파일 탭에서 eye_preprocess.py 를 /content 에 올려 주세요'
sys.path.insert(0, '/content')
import eye_preprocess
importlib.reload(eye_preprocess)
from eye_preprocess import prep_eye

SIZE, SEED = 64, 0
K_OPEN, K_CLOSED = 5, 2           # 운행 전 기준 사진: 뜬 눈 5장 + 감은 눈 2장
HALF_LABEL = 1.0                  # 반쯤 감은 눈 = 경고해야 하는 눈 (11/05)
EPOCHS, STEPS, BATCH_EP = 6, 1200, 64   # 에포크마다 문제 1200 묶음 × 64개
LR, NUM_WORKERS = 3e-4, 2
MODES = ['none', 'open', 'open_closed']
CACHE548 = '/content/drive/MyDrive/sleep_detector/aihub548_eye'     # 본인 드라이브에만 (7-1 과 같은 파일)
TRAIN_DIR = '/content/drive/MyDrive/sleep_detector/eye_cnn'
OUT = f'{TRAIN_DIR}/personal'
os.makedirs(OUT, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '← 런타임 유형을 T4 GPU 로')

S548 = json.load(open(f'{CACHE548}/summary.json', encoding='utf-8'))
parts = [np.load(f'{CACHE548}/{n}') for n in S548['chunks']]
X = np.concatenate([z['X'] for z in parts])                       # 96×96 흑백 uint8
meta = pd.DataFrame({k: np.concatenate([z[k] for z in parts]) for k in ('fname', 'side', 'y', 'driver', 'split', 'view')})
del parts
meta['cls'] = meta['y'].map({0: 'open', 1: 'closed', -1: 'half'})
meta['target'] = meta['y'].map({0: 0.0, 1: 1.0, -1: HALF_LABEL}).astype(np.float32)
meta['g'] = meta['driver'].astype(str) + '|' + meta['side'].astype(str)
assert X.ndim == 3, f'흑백 눈이어야 함 (컬러 npz X): {X.shape}'
assert not set(meta.driver[meta.split == 'train']) & set(meta.driver[meta.split == 'val']), 'train · val 에 같은 사람'
print(f'눈 {len(X)}개 · 메모리 {X.nbytes / 1e9:.2f}GB')
display(meta.groupby(['split', 'cls']).agg(눈=('y', 'size'), 사람=('driver', 'nunique')).unstack('cls'))

### 2. 사람 · 눈 쪽 묶음 + 문제 만들기

- 묶음 = 사람 × 눈 쪽 (왼 · 오) — 기준 사진은 같은 쪽 눈에서만
- 기준 뜬 눈은 **정면(F)** 에서 (앱 측정 = 정면). 정면이 모자라면 다른 방향도
- 학습 문제: 뜬 눈 50% · 감은 눈 25% · 반쯤 25% (반쯤 없는 사람은 다른 쪽에서), 문제 눈과 기준 사진은 늘 다른 사진. 증강은 문제 · 기준 따로
- val 문제: 묶음마다 기준 고정 (시드 0) + 나머지 눈 전부 = 7번 채점과 같은 사진

In [ ]:
def augment(img, rng):   # eye_cnn.ipynb 2-4 와 같음
    h, w = img.shape
    if rng.random() < 0.5:
        img = np.ascontiguousarray(img[:, ::-1])
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-10, 10), rng.uniform(0.85, 1.15))
    M[:, 2] += rng.uniform(-0.05, 0.05, 2) * (w, h)
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    img = 255.0 * (img / 255.0) ** rng.uniform(0.6, 1.6)
    if rng.random() < 0.3:
        k = int(rng.choice([3, 5])); img = cv2.GaussianBlur(img, (k, k), 0)
    if rng.random() < 0.3:
        img = img + rng.normal(0, rng.uniform(2, 8), img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)


GR = {}
for g, d in meta.groupby('g'):
    GR[g] = dict(split=d['split'].iloc[0], open=d.index[d.cls == 'open'].to_numpy(), openF=d.index[(d.cls == 'open') & (d.view == 'F')].to_numpy(),
                 closed=d.index[d.cls == 'closed'].to_numpy(), half=d.index[d.cls == 'half'].to_numpy())


def usable(g, need_closed):
    r = GR[g]
    return len(r['open']) >= K_OPEN + 1 and len(r['closed']) >= need_closed


TR = [g for g in GR if GR[g]['split'] == 'train' and usable(g, K_CLOSED)]
TR_C = [g for g in TR if len(GR[g]['closed']) >= K_CLOSED + 1]      # 감은 눈 문제를 낼 수 있는 묶음
TR_H = [g for g in TR if len(GR[g]['half'])]
print(f'학습 묶음 {len(TR)} (감은 눈 문제 {len(TR_C)} · 반쯤 문제 {len(TR_H)})')


def pick_refs(g, rng, exclude=()):
    r = GR[g]
    po = [j for j in r['openF'] if j not in exclude]
    if len(po) < K_OPEN:
        po = [j for j in r['open'] if j not in exclude]
    pc = [j for j in r['closed'] if j not in exclude]
    return rng.choice(po, K_OPEN, replace=False), rng.choice(pc, K_CLOSED, replace=False)


class Episodes(torch.utils.data.Dataset):     # 학습 — 부를 때마다 새 문제
    def __len__(self):
        return STEPS * BATCH_EP
    def __getitem__(self, i):
        rng = np.random.default_rng(int(torch.randint(0, 2**31, ()).item()))
        u = rng.random()
        if u < 0.5:
            g = TR[rng.integers(len(TR))]; q = rng.choice(GR[g]['open'])
        elif u < 0.75:
            g = TR_C[rng.integers(len(TR_C))]; q = rng.choice(GR[g]['closed'])
        else:
            g = TR_H[rng.integers(len(TR_H))]; q = rng.choice(GR[g]['half'])
        o, c = pick_refs(g, rng, exclude=(q,))
        f = lambda j: prep_eye(augment(X[j], rng), SIZE)
        return (torch.from_numpy(f(q))[None], torch.from_numpy(np.stack([f(j) for j in o]))[:, None],
                torch.from_numpy(np.stack([f(j) for j in c]))[:, None], torch.tensor(meta.target.iat[q]))


# val — 묶음마다 기준 고정 (시드 0), 나머지 눈 전부가 문제
rng0 = np.random.default_rng(SEED)
VAL_REF, VAL_Q = {}, []
for g in sorted(GR):
    if GR[g]['split'] == 'val' and usable(g, K_CLOSED):
        o, c = pick_refs(g, rng0)
        VAL_REF[g] = (o, c)
        used = set(o) | set(c)
        VAL_Q += [(g, j) for k in ('open', 'closed', 'half') for j in GR[g][k] if j not in used]
REF_PHOTOS = set(meta.fname.iloc[np.concatenate([np.r_[o, c] for o, c in VAL_REF.values()])])
print(f'val 묶음 {len(VAL_REF)} · 문제 눈 {len(VAL_Q)} · 기준 사진 {len(REF_PHOTOS)}장 (채점에서 뺌)')
PV = torch.from_numpy(np.stack([prep_eye(X[j], SIZE) for j in np.flatnonzero(meta.split.to_numpy() == 'val')]))  # val 눈 전부 (증강 없음)
VPOS = {j: k for k, j in enumerate(np.flatnonzero(meta.split.to_numpy() == 'val'))}

### 3. 모델 — ResNet18 로 사진마다 특징 → (지금 눈, 내 뜬 눈 평균, 내 감은 눈 평균, 차이) → 감음 쪽 logit

In [ ]:
class RefNet(torch.nn.Module):
    def __init__(self, mode):
        super().__init__()
        self.mode = mode
        self.enc = torchvision.models.resnet18(weights='IMAGENET1K_V1')
        self.enc.fc = torch.nn.Identity()
        n = {'none': 1, 'open': 3, 'open_closed': 5}[mode]
        self.head = torch.nn.Sequential(torch.nn.Linear(512 * n, 256), torch.nn.ReLU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1))

    def embed(self, x):                      # (N, 1, 64, 64) → (N, 512)
        return self.enc(x.expand(-1, 3, -1, -1))

    def combine(self, fq, fo=None, fc=None):
        feats = [fq]
        if self.mode != 'none':
            feats += [fo, (fq - fo).abs()]
        if self.mode == 'open_closed':
            feats += [fc, (fq - fc).abs()]
        return self.head(torch.cat(feats, 1)).squeeze(1)

    def forward(self, q, o, c):
        B = q.shape[0]
        if self.mode == 'none':
            return self.combine(self.embed(q))
        imgs = torch.cat([q, o.flatten(0, 1)] + ([c.flatten(0, 1)] if self.mode == 'open_closed' else []))
        f = self.embed(imgs)
        fq, fo = f[:B], f[B:B + B * K_OPEN].view(B, K_OPEN, -1).mean(1)
        fc = f[B + B * K_OPEN:].view(B, K_CLOSED, -1).mean(1) if self.mode == 'open_closed' else None
        return self.combine(fq, fo, fc)


@torch.no_grad()
def val_probs(m, bs=1024):
    # val 눈 전부를 한 번씩 특징으로 → 묶음마다 기준 평균 → 문제마다 확률 (VAL_Q 순서)
    m.eval(); F = []
    for i in range(0, len(PV), bs):
        with torch.autocast(device, enabled=(device == 'cuda')):
            F.append(m.embed(PV[i:i + bs, None].to(device)).float())
    F = torch.cat(F)
    proto = {g: (F[[VPOS[j] for j in o]].mean(0), F[[VPOS[j] for j in c]].mean(0)) for g, (o, c) in VAL_REF.items()}
    out = []
    for i in range(0, len(VAL_Q), 4096):
        chunk = VAL_Q[i:i + 4096]
        fq = F[[VPOS[j] for _, j in chunk]]
        fo = torch.stack([proto[g][0] for g, _ in chunk]); fc = torch.stack([proto[g][1] for g, _ in chunk])
        out.append(torch.sigmoid(m.combine(fq, fo, fc)).cpu())
    return torch.cat(out).numpy()


VT = meta.target.to_numpy()[[j for _, j in VAL_Q]]
print('모델 3개:', MODES)

### 4. 학습 — 모델 3개 (끊기면 이 셀 다시, 끝난 에포크 · 모델은 건너뜀)

In [ ]:
def bce(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def train(mode):
    out = f'{OUT}/eye_personal_{mode}_s{SEED}.pt'
    if os.path.exists(out):
        print('끝난 모델 (건너뜀):', out); return
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = RefNet(mode).to(device)
    o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS)
    sc = torch.amp.GradScaler(enabled=(device == 'cuda'))
    lossf = torch.nn.BCEWithLogitsLoss()
    last, best = f'{OUT}/{mode}_s{SEED}_last.pt', f'{OUT}/{mode}_s{SEED}_best.pt'
    start, hist, best_l = 0, [], float('inf')
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        m.load_state_dict(ck['model']); o.load_state_dict(ck['opt']); s.load_state_dict(ck['sched'])
        start, hist, best_l = ck['epoch'] + 1, ck['hist'], ck['best']
        print(f'{mode}: {start + 1}번째 에포크부터 이어서')
    dl = torch.utils.data.DataLoader(Episodes(), batch_size=BATCH_EP, num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))
    for ep in range(start, EPOCHS):
        t0 = time.time(); m.train(); tl = n = 0
        for q, oo, cc, y in dl:
            q, oo, cc, y = (t.to(device, non_blocking=True) for t in (q, oo, cc, y))
            o.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = lossf(m(q, oo, cc).float(), y)
            sc.scale(loss).backward(); sc.step(o); sc.update()
            tl += loss.item() * len(y); n += len(y)
        s.step()
        vl = bce(val_probs(m), VT)
        hist.append(dict(epoch=ep + 1, train_loss=round(tl / n, 4), val_loss=round(vl, 4), sec=round(time.time() - t0)))
        print(mode, hist[-1])
        if vl < best_l:
            best_l = vl; torch.save(m.state_dict(), best)
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=s.state_dict(), epoch=ep, hist=hist, best=best_l), last)
    torch.save(dict(state_dict=torch.load(best, map_location='cpu', weights_only=True), mode=mode, input_size=SIZE, k_open=K_OPEN, k_closed=K_CLOSED,
                    half_label=HALF_LABEL, seed=SEED, hist=hist, data='AI-Hub 548 학습용 train (반쯤 감은 눈 포함), 사람 · 눈 쪽 기준 사진',
                    made=str(datetime.datetime.now())[:16]), out)
    print('저장:', out)


for mode in MODES:
    train(mode)

### 5. 채점 — 548 학습용 val 115명, 사진 단위 (두 눈 평균), 같은 오경보에서 반쯤 · 감은 눈을 감음 쪽으로 못 본 비율

- 감음 쪽 = 확률 ≥ 기준값. **오경보** = 뜬 눈인데 감음 쪽 / **반쯤 못 봄** = 반쯤 감은 사진인데 감음 쪽이 아님 (서버라면 취소) / **감은 눈 못 봄** = 감았는데 감음 쪽이 아님 — 셋 다 따로
- 기준값을 0 ~ 1 로 훑어 **오경보 ≤ 1 · 2 · 3 · 5% 가 되는 가장 낮은 기준값** 에서 잼 (CNN 만, EAR 없음 — 서버 규칙 비교는 효과 있을 때 PC 에서)
- 지금 서버 모델도 같은 사진 · 같은 방식으로 (기준 사진은 안 씀)

In [ ]:
def photo_level(p):
    d = pd.DataFrame({'fname': meta.fname.to_numpy()[[j for _, j in VAL_Q]], 'cls': meta.cls.to_numpy()[[j for _, j in VAL_Q]],
                      'driver': meta.driver.to_numpy()[[j for _, j in VAL_Q]], 'p': p})
    d = d[~d.fname.isin(REF_PHOTOS)]
    return d.groupby('fname').agg(cls=('cls', 'first'), driver=('driver', 'first'), p=('p', 'mean'), n=('p', 'size')).reset_index()


def at_fa(ph, targets=(0.01, 0.02, 0.03, 0.05)):
    rows = {}
    for fa in targets:
        best = None
        for t in np.linspace(0, 1, 2001):
            a = ph.p >= t
            f = a[ph.cls == 'open'].mean()
            if f <= fa:
                best = (t, f, 1 - a[ph.cls == 'half'].mean(), 1 - a[ph.cls == 'closed'].mean()); break
        rows[f'오경보≤{fa:.0%}'] = '—' if best is None else f'반쯤 못 봄 {best[2]:.1%} · 감은 눈 못 봄 {best[3]:.1%} (t {best[0]:.3f})'
    return rows


@torch.no_grad()
def cnn_probs(path):
    ck = torch.load(path, map_location='cpu', weights_only=True)
    net = torchvision.models.resnet18(); net.fc = torch.nn.Linear(512, 1)
    net.load_state_dict(ck['state_dict']); net.to(device).eval()
    p = []
    for i in range(0, len(PV), 1024):
        p.append(torch.sigmoid(net(PV[i:i + 1024, None].to(device).expand(-1, 3, -1, -1)).float().squeeze(1)).cpu())
    p = torch.cat(p).numpy()
    return np.array([p[VPOS[j]] for _, j in VAL_Q])


RES = {}
for mode in MODES:
    m = RefNet(mode).to(device)
    m.load_state_dict(torch.load(f'{OUT}/eye_personal_{mode}_s{SEED}.pt', map_location=device, weights_only=True)['state_dict'])
    RES[mode] = photo_level(val_probs(m))
for name in ('eye_cnn12_k100_100_s0.pt', 'eye_cnn_548.pt'):   # 지금 서버 · 12번 기준 (있으면)
    if os.path.exists(f'{TRAIN_DIR}/{name}'):
        RES[name] = photo_level(cnn_probs(f'{TRAIN_DIR}/{name}'))
    else:
        print('없음 (건너뜀):', f'{TRAIN_DIR}/{name}')

ph = next(iter(RES.values()))
print(f"채점 사진: 뜸 {(ph.cls == 'open').sum()} · 반쯤 {(ph.cls == 'half').sum()} · 감음 {(ph.cls == 'closed').sum()} (사람 {ph.driver.nunique()}명, 기준 사진 뺌)")
TABLE = pd.DataFrame({k: at_fa(v) for k, v in RES.items()}).T
display(TABLE)
TABLE.to_csv(f'{OUT}/result_val115_s{SEED}.csv', encoding='utf-8-sig')

### 6. 판정 (0번 규칙 3 그대로 — 손으로 바꾸지 않기)

In [ ]:
def half_closed_at(ph, fa=0.03):
    for t in np.linspace(0, 1, 2001):
        a = ph.p >= t
        if a[ph.cls == 'open'].mean() <= fa:
            return 1 - a[ph.cls == 'half'].mean(), 1 - a[ph.cls == 'closed'].mean()


h_none, c_none = half_closed_at(RES['none'])
for mode in ('open', 'open_closed'):
    h, c = half_closed_at(RES[mode])
    print(f'{mode:<12} 오경보 3%: 반쯤 못 봄 {h:.1%} (none {h_none:.1%}, {100 * (h - h_none):+.1f}%p) · 감은 눈 못 봄 {c:.1%} (none {c_none:.1%})')
h, c = half_closed_at(RES['open_closed'])
ok = (h_none - h) > 0.02 and c <= c_none
print('\n판정:', '효과 있음 → 팀에 제안 (앱 측정에 눈 감기 추가 · 서버 기준 사진 보관) + 새 데이터 최종 시험 1회' if ok
      else '효과 없음 → 지금 방식 유지, 결과만 보고')
json.dump(dict(table=TABLE.to_dict(), decision=bool(ok), made=str(datetime.datetime.now())[:16]), open(f'{OUT}/decision_s{SEED}.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)

### 7. 결과 (11/05 Colab T4 — 사용자 실행, 판정은 0번 규칙 3 그대로)

**판정: 효과 없음 → 지금 방식 유지, 결과만 보고**

- 채점: 548 학습용 val 중 기준 사진을 만들 수 있는 **83명** (같은 쪽 눈에 뜬 눈 6장 · 감은 눈 2장 이상), 사진 단위 (두 눈 평균), 기준 사진 뺌 — 뜸 4,304 · 반쯤 307 · 감음 387
- 같은 오경보에서 **반쯤 감은 사진을 감음 쪽으로 못 본 비율 · 감은 눈 못 봄** (CNN 만, EAR 없음 — 서버에서는 '취소' 에 해당)

| 모델 | 오경보 ≤ 1% | ≤ 2% | **≤ 3% (판정)** | ≤ 5% |
|---|---|---|---|---|
| `none` (지금 눈만, 반쯤 학습) | 46.3% · 3.6% | 33.2% · 2.1% | **21.8% · 1.6%** | 13.0% · 1.3% |
| `open` (+ 내 뜬 눈 5) | 48.9% · 4.9% | 30.3% · 2.1% | 18.9% · 1.0% | 9.1% · 0.8% |
| `open_closed` (+ 내 감은 눈 2) | 46.3% · 4.4% | 29.0% · 2.3% | **24.1% · 2.1%** | 15.0% · 1.0% |
| (참고) `eye_cnn12_k100_100_s0` | 47.9% · 3.1% | 33.6% · 1.8% | 25.4% · 0.8% (t 0.063) | 18.6% · 0.8% |
| (참고) `eye_cnn_548` (지금 서버) | 55.7% · 4.1% | 38.4% · 1.3% | 30.0% · 0.8% (t 0.082) | 16.0% · 0.8% |

- 규칙 3: `open_closed` 가 오경보 3% 에서 `none` 보다 반쯤 못 봄이 2%p 넘게 낮아야 함 → **+2.3%p (오히려 나쁨), 감은 눈 못 봄도 1.6 → 2.1%** → 효과 없음
- `open` 은 3% 에서 −2.9%p 지만 미리 정한 비교 대상이 아니고 1% 에서는 +2.6%p (방향이 오경보마다 바뀜) → **'경향' 으로만**, 고르지 않음
- 숫자 시뮬레이션 (`personal_baseline_sim.py`, 시드 5개) 과 같은 결론: **같은 사람의 기준 사진을 줘도 반쯤 감은 눈 판단이 뚜렷이 나아지지 않음** → 발표에 "시도했지만 효과 없음"
- 덤: 반쯤 감은 눈을 학습에 넣은 `none` 이 같은 오경보에서 서버 모델보다 반쯤 못 봄이 낮음 (21.8% vs `eye_cnn_548` 30.0%) — 대신 감은 눈 못 봄 1.6% vs 0.8%. 서버 모델들은 t 가 0.06 ~ 0.08 까지 내려가야 오경보 3% (눈 단위 · CNN 만이라 서버 규칙과 다름) → **판단은 서버 규칙으로 재는 13번 (`half1` · `half05`) 에서**
- 이 노트북 셀 출력의 '놓침' 문구는 실행 당시 버전 (11/05 팀장 문구로 고치기 전) — 뜻은 '감음 쪽으로 못 봄' 과 같음